# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/usr/bin/python3


In [2]:
import sys
import os

# Using tf_keras (legacy Keras 2) instead of downgrading keras
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0" tf_keras -q

os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

import tf_keras as keras

Sequential = keras.Sequential
Dense = keras.layers.Dense
LSTM = keras.layers.LSTM
to_categorical = keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.20.0
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [4]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [5]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
X = wine.data
y = wine.target

In [6]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [7]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
num_classes = 3

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat  = to_categorical(y_test,  num_classes=num_classes)

In [9]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [10]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(X_train, y_train_cat,
                    epochs=20,
                    batch_size=8,
                    validation_split=0.2)

Epoch 1/20
13/13 [==============================] - 2s 35ms/step - loss: 0.9103 - accuracy: 0.6768 - val_loss: 0.7568 - val_accuracy: 0.9200
Epoch 2/20
13/13 [==============================] - 0s 15ms/step - loss: 0.6611 - accuracy: 0.8889 - val_loss: 0.5446 - val_accuracy: 0.9200
Epoch 3/20
13/13 [==============================] - 0s 14ms/step - loss: 0.4879 - accuracy: 0.9394 - val_loss: 0.3996 - val_accuracy: 0.9200
Epoch 4/20
13/13 [==============================] - 0s 10ms/step - loss: 0.3514 - accuracy: 0.9596 - val_loss: 0.2981 - val_accuracy: 0.9200
Epoch 5/20
13/13 [==============================] - 0s 15ms/step - loss: 0.2561 - accuracy: 0.9697 - val_loss: 0.2324 - val_accuracy: 0.9200
Epoch 6/20
13/13 [==============================] - 0s 11ms/step - loss: 0.1868 - accuracy: 0.9798 - val_loss: 0.1923 - val_accuracy: 0.9200
Epoch 7/20
13/13 [==============================] - 0s 14ms/step - loss: 0.1423 - accuracy: 0.9899 - val_loss: 0.1630 - val_accuracy: 0.9200
Epoch 8/20
13

In [11]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
y_prob = model.predict(X_test)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print("Accuracy:", np.mean(y_pred == y_true))
print("\nClassification Report:\n", classification_report(y_true, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 11ms/step
Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
 [[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


In [12]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Base TFLite model size: {os.path.getsize('model_base.tflite') / 1024:.2f} KB")

Base TFLite model size: 14.01 KB


## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [20]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]

def file_size_kb(filename):
    return os.path.getsize(filename) / 1024

def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.
        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8


    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]


    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    # <-- Enter your code here <--#

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # <-- Enter your code here for TFLite inference <--#
    tflite_model = converter.convert()

    with open(filename, "wb") as f:
        f.write(tflite_model)

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_index = input_details[0]["index"]
    output_index = output_details[0]["index"]
    input_dtype = input_details[0]["dtype"]
    output_dtype = output_details[0]["dtype"]
    input_scale, input_zero_point = input_details[0].get("quantization", (0.0, 0))
    output_scale, output_zero_point = output_details[0].get("quantization", (0.0, 0))

    y_pred = []
    for sample in X_test:
        x = sample.astype(np.float32)[None, :]
        if input_dtype in [np.int8, np.uint8] and input_scale not in (0, 0.0):
            x = np.round(x / input_scale + input_zero_point).astype(input_dtype)

        interpreter.set_tensor(input_index, x)
        interpreter.invoke()
        out = interpreter.get_tensor(output_index)

        if output_dtype in [np.int8, np.uint8] and output_scale not in (0, 0.0):
            out = (out.astype(np.float32) - output_zero_point) * output_scale

        y_pred.append(np.argmax(out, axis=1)[0])

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    # <-- Enter your code here: print classification_report and confusion_matrix <--#
    print("\nClassification Report:\n", classification_report(y_true, y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))



In [21]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
quantize_and_evaluate(model, X_test, y_test_cat, 'int8', 'model_int8.tflite')
quantize_and_evaluate(model, X_test, y_test_cat, 'float16', 'model_float16.tflite')
quantize_and_evaluate(model, X_test, y_test_cat, 'dynamic', 'model_dynamic.tflite')

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



INT8 TFLite model size: 7.91 KB

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
 [[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



FLOAT16 TFLite model size: 8.77 KB

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
 [[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]

DYNAMIC TFLite model size: 8.44 KB

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
 [[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## Problem 1 - Part (c)

### Pruning

In [26]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
steps_per_epoch = int(np.ceil(X_train.shape[0] * 0.8 / 8))
end_step = steps_per_epoch * 10

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

In [33]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
student_model = Sequential([
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        pruning_schedule=pruning_schedule
    ),
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(3, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

In [34]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
student_model = Sequential([
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        pruning_schedule=pruning_schedule
    ),
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(3, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

In [35]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
stripped_model = tfmot.sparsity.keras.strip_pruning(student_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Pruned TFLite model size: {os.path.getsize('model_pruned.tflite') / 1024:.2f} KB")

Pruned TFLite model size: 13.36 KB


In [36]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred = np.argmax(stripped_model.predict(X_test), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 9ms/step
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        19
           1       0.11      0.05      0.07        21
           2       0.18      0.57      0.28        14

    accuracy                           0.17        54
   macro avg       0.10      0.21      0.11        54
weighted avg       0.09      0.17      0.10        54

[[ 0  3 16]
 [ 0  1 20]
 [ 1  5  8]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [22]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
steps_per_epoch = int(np.ceil(X_train.shape[0] * 0.8 / 8))
end_step = steps_per_epoch * 10

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

In [24]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
teacher_soft_labels = model.predict(X_train)

4/4 [==============================] - 0s 6ms/step


In [32]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#
teacher_preds_soft = model.predict(X_train)
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)

def distillation_loss(y_true_combined, y_pred):

    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
  y_true_hard = y_true_combined[:, :3]
  y_true_soft = y_true_combined[:, 3:]

  hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
  soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

  alpha = 0.5
  return alpha * hard_loss + (1 - alpha) * soft_loss

4/4 [==============================] - 0s 2ms/step


In [38]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#

student_model.compile(optimizer='adam', loss=distillation_loss, metrics=['accuracy'])

history = student_model.fit(
    X_train,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()]
)

Epoch 1/10
13/13 [==============================] - 6s 60ms/step - loss: 1.1029 - accuracy: 0.3737 - val_loss: 0.9563 - val_accuracy: 0.6800
Epoch 2/10
13/13 [==============================] - 0s 8ms/step - loss: 0.8023 - accuracy: 0.8687 - val_loss: 0.7002 - val_accuracy: 0.8400
Epoch 3/10
13/13 [==============================] - 0s 9ms/step - loss: 0.5917 - accuracy: 0.9596 - val_loss: 0.5139 - val_accuracy: 0.8400
Epoch 4/10
13/13 [==============================] - 0s 8ms/step - loss: 0.4354 - accuracy: 0.9596 - val_loss: 0.3773 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 9ms/step - loss: 0.3134 - accuracy: 0.9798 - val_loss: 0.2796 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 9ms/step - loss: 0.2257 - accuracy: 0.9798 - val_loss: 0.2167 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 8ms/step - loss: 0.1682 - accuracy: 0.9899 - val_loss: 0.1749 - val_accuracy: 0.9600
Epoch 8/10
13/13 [=

In [39]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_model)

print(f"KD TFLite model size: {os.path.getsize('model_kd.tflite') / 1024:.2f} KB")

KD TFLite model size: 29.50 KB


In [43]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred = np.argmax(student_model.predict(X_test), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 16ms/step
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        19
           1       0.95      1.00      0.98        21
           2       1.00      1.00      1.00        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

[[18  1  0]
 [ 0 21  0]
 [ 0  0 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [ ]:
# <-- (if needed) Enter your code here <--#

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
